# DOSE — YOLOv12n Training (150 epochs)

YOLOv12n (2.6M params) on 9,502 Vietnamese pill images (108 classes).

AdamW optimizer, mixup=0.2, copy_paste=0.1 for better augmentation.

In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.2

In [2]:
from ultralytics import YOLO
import os, shutil, json, random, glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATASET_PATH = "/kaggle/input/datasets/ganggangtypeshit/dose-yolo-dataset"

# Load class names
with open(os.path.join(DATASET_PATH, "class_names.json")) as f:
    class_names_dict = json.load(f)
class_names_list = [class_names_dict[str(i)] for i in range(108)]
NC = len(class_names_list)
print(f"Loaded {NC} class names")

# Patch YAML path for Kaggle
yaml_path = os.path.join(DATASET_PATH, "dataset.yaml")
with open(yaml_path) as f:
    content = f.read()
content = content.replace("path: /home/lducc/code/dose/data/yolo", f"path: {DATASET_PATH}")
working_yaml = "/kaggle/working/dataset.yaml"
with open(working_yaml, "w") as f:
    f.write(content)
print("Dataset ready:", working_yaml)

# Verify dataset
train_count = len(list(Path(DATASET_PATH).glob("images/train/*.jpg")))
val_count = len(list(Path(DATASET_PATH).glob("images/val/*.jpg")))
print(f"Train: {train_count} images, Val: {val_count} images")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loaded 108 class names
Dataset ready: /kaggle/working/dataset.yaml
Train: 8551 images, Val: 951 images


In [3]:
# YOLOv12n — 150 epochs, AdamW, augmented
model = YOLO("yolo12n.pt")

results = model.train(
    data=working_yaml,
    epochs=150,
    imgsz=640,
    batch=32,
    patience=25,
    device=0,
    name="dose_yolo12n_200ep",
    exist_ok=True,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.001,
    cos_lr=True,
    mixup=0.2,
    copy_paste=0.1,
    mosaic=1.0,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    close_mosaic=15,
    warmup_epochs=3.0,
)

Ultralytics 8.4.81 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dose_yolo12n_200ep, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True

In [4]:
model.export(format="onnx", imgsz=640, half=True)

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.81 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv12n summary (fused): 159 layers, 2,624,588 parameters, 0 gradients, 6.7 GFLOPs

PyTorch: starting from '/kaggle/working/runs/detect/dose_yolo12n_200ep/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 112, 8400) (5.4 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 226ms
 Downloaded onnxruntime
Prepared 2 packages in 273ms
Installed 2 packages in 11ms
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to tak

'/kaggle/working/runs/detect/dose_yolo12n_200ep/weights/best.onnx'

In [5]:
# Find and copy ONNX
onnx_src = glob.glob("/kaggle/working/runs/detect/dose_yolo12*/weights/best.onnx")
if onnx_src:
    shutil.copy(onnx_src[0], "/kaggle/working/vaipe.onnx")
    print(f"ONNX: {onnx_src[0]} -> /kaggle/working/vaipe.onnx")
    print(f"Size: {os.path.getsize('/kaggle/working/vaipe.onnx') / 1e6:.1f} MB")
else:
    print("ONNX not found! Check runs/")

ONNX: /kaggle/working/runs/detect/dose_yolo12n_200ep/weights/best.onnx -> /kaggle/working/vaipe.onnx
Size: 5.5 MB


In [6]:
csv_files = glob.glob("/kaggle/working/runs/detect/dose_yolo12*/results.csv")
for f in csv_files:
    dst = os.path.join("/kaggle/working", os.path.basename(f))
    shutil.copy(f, dst)
    print(f"Copied: {dst}")

Copied: /kaggle/working/results.csv


In [7]:
# Benchmark: validation metrics
# Find best weights
best_weights = glob.glob("/kaggle/working/runs/detect/dose_yolo12*/weights/best.pt")
model = YOLO(best_weights[0])
print(f"Loading: {best_weights[0]}")

metrics = model.val(data=working_yaml, split="val")

print(f"\n{'='*50}")
print(f"BENCHMARK RESULTS")
print(f"{'='*50}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"{'='*50}")

# Per-class metrics — ap is always length nc
nc = 108

# metrics.box.ap = per-class mAP50 (always length nc)
# metrics.box.ap50 = same as ap for mAP50
ap50 = metrics.box.ap.tolist() if hasattr(metrics.box, "ap") else [0.0] * nc
ap50 = ap50[:nc]  # truncate if longer
while len(ap50) < nc:
    ap50.append(0.0)

# p and r may be shorter — safe-pad them
def safe_pad(arr, target_len):
    """Convert to list, handle nested arrays, pad to target length."""
    if arr is None or (hasattr(arr, '__len__') and len(arr) == 0):
        return [0.0] * target_len
    if hasattr(arr, 'tolist'):
        arr = arr.tolist()
    result = []
    for x in arr[:target_len]:
        if isinstance(x, (list, tuple, np.ndarray)):
            result.append(float(np.mean(x)))
        else:
            result.append(float(x))
    while len(result) < target_len:
        result.append(0.0)
    return result

p_vals = safe_pad(getattr(metrics.box, "p", None), nc)
r_vals = safe_pad(getattr(metrics.box, "r", None), nc)

df = pd.DataFrame({
    "class_id": range(nc),
    "name": class_names_list,
    "mAP50": ap50,
    "precision": p_vals,
    "recall": r_vals,
})
per_class_path = "/kaggle/working/per_class_metrics.csv"
df.to_csv(per_class_path, index=False)
print(f"\nSaved: {per_class_path}")

# Top 10 best and worst classes
df_sorted = df.sort_values("mAP50", ascending=False)
print(f"\nTop 10 classes (by mAP50):")
for _, row in df_sorted.head(10).iterrows():
    print(f"  {row['mAP50']:.3f}  {row['name']}")
print(f"\nBottom 10 classes (by mAP50):")
for _, row in df_sorted.tail(10).iterrows():
    print(f"  {row['mAP50']:.3f}  {row['name']}")


Loading: /kaggle/working/runs/detect/dose_yolo12n_200ep/weights/best.pt
Ultralytics 8.4.81 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv12n summary (fused): 159 layers, 2,624,588 parameters, 0 gradients, 6.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 30.7±5.5 MB/s, size: 27.8 KB)
val: Scanning /kaggle/input/datasets/ganggangtypeshit/dose-yolo-dataset/labels/val... 951 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 951/951 552.2it/s 1.7s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/ganggangtypeshit/dose-yolo-dataset/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 5.5it/s 10.8s
                   all        951       3256      0.772       0.57      0.669      0.467
paracetamol 500mg 500mg          5          5      0.804        0.6      0.631       0.46
  troysar am 5mg +50mg         11         14      0.582      0.429      0.425 

In [8]:
# Energy-Based OOD (Out-of-Distribution) Detection
# Tunable thresholds — adjust per your needs

REJECT_CONF_THRESHOLD = 0.50    # max prob below this → reject
REJECT_ENERGY_THRESHOLD = -15.0 # energy above this → reject (lower = more negative = better)
REJECT_ENTROPY_THRESHOLD = 2.5  # entropy above this → reject (more uncertain)

def compute_energy(probs):
    logits = np.log(probs + 1e-10)
    return -np.log(np.sum(np.exp(logits)))

def compute_entropy(probs):
    return -np.sum(probs * np.log(probs + 1e-10))

def classify_detection(conf, pred_class, max_prob, energy, entropy):
    if pred_class == 107:
        return "reject", "out_of_prescription"
    if max_prob < REJECT_CONF_THRESHOLD:
        return "reject", f"low_confidence ({max_prob:.3f})"
    if energy > REJECT_ENERGY_THRESHOLD:
        return "reject", f"unknown_pill (energy={energy:.2f})"
    if entropy > REJECT_ENTROPY_THRESHOLD:
        return "reject", f"ambiguous (entropy={entropy:.2f})"
    return "accept", "ok"

print("OOD thresholds:")
print(f"  Confidence:  {REJECT_CONF_THRESHOLD}")
print(f"  Energy:      {REJECT_ENERGY_THRESHOLD}")
print(f"  Entropy:     {REJECT_ENTROPY_THRESHOLD}")
print("Adjust these and re-run this cell to tune rejection.")

OOD thresholds:
  Confidence:  0.5
  Energy:      -15.0
  Entropy:     2.5
Adjust these and re-run this cell to tune rejection.


In [9]:
# OOD benchmark: test rejection on val set
val_dir = Path(DATASET_PATH) / "images" / "val"
val_images = sorted(val_dir.glob("*.jpg"))

reject_stats = {"accept": 0, "reject": 0, "reject_reasons": {}}
ood_scores = []

for img_path in val_images[:200]:  # test on first 200 for speed
    r = model(str(img_path), verbose=False)[0]
    if len(r.boxes) == 0:
        reject_stats["reject"] += 1
        reject_stats["reject_reasons"]["no_detections"] = \
            reject_stats["reject_reasons"].get("no_detections", 0) + 1
        continue
    
    probs = r.probs if r.probs is not None else None
    if probs is None:
        # Fall back to box-based confidence
        confs = r.boxes.conf.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy().astype(int)
        for conf, cls in zip(confs, classes):
            max_prob = float(conf)
            pred_class = int(cls)
            # Approximate energy/entropy from single confidence
            energy = -np.log(max_prob + 1e-10)
            entropy = -max_prob * np.log(max_prob + 1e-10) - (1-max_prob) * np.log(1-max_prob + 1e-10)
            decision, reason = classify_detection(max_prob, pred_class, max_prob, energy, entropy)
            reject_stats[decision] += 1
            if decision == "reject":
                reject_stats["reject_reasons"][reason.split("(")[0].strip()] = \
                    reject_stats["reject_reasons"].get(reason.split("(")[0].strip(), 0) + 1
            ood_scores.append({"energy": energy, "entropy": entropy, "max_prob": max_prob})

total = reject_stats["accept"] + reject_stats["reject"]
print(f"OOD Analysis (first 200 val images):")
print(f"  Accepted: {reject_stats['accept']}/{total} ({reject_stats['accept']/max(total,1)*100:.1f}%)")
print(f"  Rejected: {reject_stats['reject']}/{total} ({reject_stats['reject']/max(total,1)*100:.1f}%)")
if reject_stats["reject_reasons"]:
    print(f"  Rejection reasons:")
    for reason, count in sorted(reject_stats["reject_reasons"].items(), key=lambda x: -x[1]):
        print(f"    {reason}: {count}")
if ood_scores:
    energies = [s["energy"] for s in ood_scores]
    entropies = [s["entropy"] for s in ood_scores]
    probs_list = [s["max_prob"] for s in ood_scores]
    print(f"\nScore distributions:")
    print(f"  Energy:  mean={np.mean(energies):.2f}, std={np.std(energies):.2f}")
    print(f"  Entropy: mean={np.mean(entropies):.2f}, std={np.std(entropies):.2f}")
    print(f"  Conf:    mean={np.mean(probs_list):.3f}, std={np.std(probs_list):.3f}")

OOD Analysis (first 200 val images):
  Accepted: 0/717 (0.0%)
  Rejected: 717/717 (100.0%)
  Rejection reasons:
    unknown_pill: 476
    out_of_prescription: 178
    low_confidence: 55
    no_detections: 8

Score distributions:
  Energy:  mean=0.28, std=0.26
  Entropy: mean=0.46, std=0.14
  Conf:    mean=0.781, std=0.158


In [10]:
# Full val inference: score every image
val_dir = Path(DATASET_PATH) / "images" / "val"
val_images = sorted(val_dir.glob("*.jpg"))
print(f"Running inference on {len(val_images)} validation images...")

results_all = model(list(map(str, val_images)), verbose=False)

image_scores = []
for img_path, r in zip(val_images, results_all):
    confs = r.boxes.conf.cpu().numpy() if len(r.boxes) > 0 else np.array([0.0])
    image_scores.append({
        "path": img_path,
        "avg_conf": float(np.mean(confs)),
        "n_boxes": len(r.boxes),
    })

# Select 16 random + 16 worst-performing (lowest avg conf)
random.seed(42)
random_16 = random.sample(image_scores, 16)
worst_16 = sorted(image_scores, key=lambda x: x["avg_conf"])[:16]

print(f"Random samples avg confidence: {np.mean([s['avg_conf'] for s in random_16]):.3f}")
print(f"Worst samples avg confidence: {np.mean([s['avg_conf'] for s in worst_16]):.3f}")


Running inference on 951 validation images...


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.35 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.22 GiB is free. Including non-PyTorch memory, this process has 13.34 GiB memory in use. Of the allocated memory 11.68 GiB is allocated by PyTorch, and 1.46 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
def plot_grid(samples, title, save_path):
    fig, axes = plt.subplots(4, 4, figsize=(20, 20))
    fig.suptitle(title, fontsize=16, y=1.02)
    for ax, s in zip(axes.flatten(), samples):
        r = model(str(s["path"]), verbose=False)[0]
        plotted = r.plot(labels=True, conf=True)
        ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{s['path'].name}\nconf={s['avg_conf']:.2f}, n={s['n_boxes']}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

plot_grid(random_16, "16 Random Validation Samples", "/kaggle/working/inference_random.png")


In [ ]:
plot_grid(worst_16, "16 Worst-Performing Samples (Lowest Avg Confidence)", "/kaggle/working/inference_worst.png")
